<a href="https://colab.research.google.com/github/Rheaxu/english-writing-sheet/blob/main/CVC_Website_Generator_EdgeTTS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CVC Worksheet to Interactive Website Generator

This notebook turns any CVC first-and-last-letter worksheet PDF into a ready-to-use interactive website.

**How to use:**
1. Run all cells top-to-bottom (Runtime → Run all, or Shift+Enter)
2. Upload your PDF when prompted
3. Confirm the word list for each page
4. Download the finished zip at the end

---


## Step 1 — Install dependencies
*(Run once per session)*

In [ ]:
print("Installing packages...")
import subprocess, sys
pkgs = ["pypdf", "pdf2image", "pdfplumber", "pillow", "edge-tts", "pydub"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkgs, check=True)
subprocess.run(["apt-get", "install", "-y", "-q", "tesseract-ocr", "poppler-utils"], check=True)
print("\n All dependencies installed.")


## Step 2 — Upload your PDF

In [ ]:
from google.colab import files as colab_files
print("Choose your CVC worksheet PDF:")
uploaded = colab_files.upload()
if uploaded:
    PDF_PATH = list(uploaded.keys())[0]
    print(f"\nLoaded: {PDF_PATH}")
else:
    raise FileNotFoundError("No file uploaded — re-run this cell and choose a PDF.")


## Step 3 — Preview PDF pages

In [ ]:
from pdf2image import convert_from_path
from IPython.display import display

pages = convert_from_path(PDF_PATH, dpi=130)
print(f"Found {len(pages)} page(s)\n")
for i, page in enumerate(pages):
    thumb = page.copy()
    thumb.thumbnail((380, 520))
    display(thumb)
    print(f"  Page {i+1}")


## Step 4 — Extract clipart images

Each page's images are listed in top-to-bottom order — they should match your word list.


In [ ]:
from pypdf import PdfReader
from PIL import Image as PILImage
import io, os
from IPython.display import display

reader = PdfReader(PDF_PATH)
os.makedirs("site/images", exist_ok=True)

page_images = []
for page_num, page in enumerate(reader.pages):
    imgs = []
    for img_obj in page.images:
        try:
            pil = PILImage.open(io.BytesIO(img_obj.data))
            imgs.append(pil)
        except Exception:
            pass
    page_images.append(imgs)
    print(f"Page {page_num+1}: {len(imgs)} image(s)")

print(f"\nTotal: {sum(len(x) for x in page_images)} images")
print("\nPreview — first 3 images from page 1:")
for img in page_images[0][:3]:
    t = img.copy(); t.thumbnail((130, 130)); display(t)


## Step 5 — Read word list via OCR

Reads the "WORDS: ..." line at the bottom of each page. You can correct mistakes in Step 6.


In [ ]:
!apt-get install -y tesseract-ocr
!pip install pytesseract

import pytesseract, re

ocr_words_per_page = []
for page_num, page_img in enumerate(pages):
    w, h = page_img.size
    strip = page_img.crop((0, int(h * 0.82), w, h))
    raw = pytesseract.image_to_string(strip, config="--psm 6").upper()
    print(f"  Raw OCR for page {page_num+1}:\n{raw}") # Added for debugging

    m = re.search(r"WORDS?[:\s]+([A-Z,\s.]+)", raw) # Modified: allow periods in capture group
    if m:
        # Directly find words using a word boundary regex, ignoring punctuation
        words = re.findall(r"\b[A-Z]{2,5}\b", m.group(1))
    else:
        words = re.findall(r"\b[A-Z]{2,5}\b", raw)
    ocr_words_per_page.append(words)
    print(f"Page {page_num+1}: {words}")

## Step 6 — Confirm or correct the words

Edit any line below if the OCR made mistakes.  
Format: `CAB, DIG, FOG, GUM, HEN`


In [ ]:
import ipywidgets as widgets
from IPython.display import display

word_inputs = []
for i, words in enumerate(ocr_words_per_page):
    w = widgets.Text(
        value=", ".join(words),
        description=f"Page {i+1}:",
        layout=widgets.Layout(width="520px"),
        style={"description_width": "70px"}
    )
    word_inputs.append(w)
    display(w)

print("\nEdit above, then run Step 7.")


## Step 7 — Build word list and save images

In [ ]:
import re, os

confirmed_words = []  # list of (word_str, page_num)
for i, widget in enumerate(word_inputs):
    raw = widget.value.upper()
    tokens = [t.strip() for t in re.split(r"[,\s]+", raw) if re.match(r"^[A-Z]{2,5}$", t.strip())]
    confirmed_words.extend([(tok.lower(), i+1) for tok in tokens])
    print(f"  Page {i+1}: {[w for w,p in confirmed_words if p==i+1]}")

print(f"\nTotal: {len(confirmed_words)} words")

# Pair images to words (zip per page)
os.makedirs("site/images", exist_ok=True)
saved = {}
for page_idx, page_imgs in enumerate(page_images):
    page_words = [w for w, p in confirmed_words if p == page_idx + 1]
    for word, img in zip(page_words, page_imgs):
        path = f"site/images/{word}.png"
        img.save(path)
        saved[word] = path
print(f"\nSaved {len(saved)} images: {list(saved.keys())}")


## Step 8 — Generate audio (edge-tts)

Creates one MP3 per phonics sound and one per full word using Microsoft Edge's free neural voices. No API key required.


In [ ]:
print("Upgrading edge-tts...")
import subprocess, sys
# Uninstall current version and install the latest
subprocess.run([sys.executable, "-m", "pip", "uninstall", "edge-tts", "-y"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "edge-tts", "-q"], check=True)
print("edge-tts upgraded. Please re-run the next cell (Step 8) to generate audio with the updated library.")

In [ ]:
import edge_tts, asyncio, os, io
from pydub import AudioSegment
from pydub.silence import split_on_silence

VOICE = "en-US-JennyNeural"

# IPA map — standard phonetic notation, tells edge_tts exactly which sound to produce
# *** Adjusted for pronounceable English approximations of phonics sounds ***
IPA = {
    "a":"ah", "e":"eh", "i":"ih", "o":"aw", "u":"uh",
    "b":"buh",  "c":"kuh",  "d":"duh",  "f":"fuh",  "g":"guh",
    "h":"huh",  "j":"juh","k":"kuh",  "l":"luh",  "m":"muh",
    "n":"nuh",  "p":"puh",  "r":"ruh",  "s":"suh",  "t":"tuh",
    "v":"vuh",  "w":"wuh",  "x":"ks", "y":"yuh",  "z":"zuh",
}

os.makedirs("site/sounds/phonics", exist_ok=True)
os.makedirs("site/sounds/words",   exist_ok=True)
os.makedirs("site/sounds/letters", exist_ok=True) # New folder for individual letter sounds

async def get_audio_bytes(communicate):
    chunks = []
    async for chunk in communicate.stream():
        if chunk["type"] == "audio":
            chunks.append(chunk["data"])
    return b"".join(chunks)

# Helper to save raw edge_tts bytes to a temporary file and load with pydub for robustness
def load_audio_from_edge_tts_bytes(data):
    # Using a temporary file to ensure pydub loads raw data correctly,
    # as BytesIO can sometimes struggle with edge_tts output's exact format.
    temp_path = "temp_audio_data.mp3"
    with open(temp_path, "wb") as f:
        f.write(data)
    audio = AudioSegment.from_file(temp_path, format="mp3")
    os.remove(temp_path)
    return audio

# New function to generate and save individual phoneme sounds
async def save_phoneme_for_char(char):
    if char not in IPA:
        print(f"Warning: No IPA mapping for character '{char}'. Skipping phoneme generation.")
        return False

    # Using English approximations of phonics sounds that edge_tts can pronounce
    data = await get_audio_bytes(edge_tts.Communicate(IPA[char], voice=VOICE))
    audio = load_audio_from_edge_tts_bytes(data)
    path = f"site/sounds/phonics/{char}.mp3"
    audio.export(path, format="mp3")
    return True

# New function to generate individual letter sounds
async def save_letter_sound(letter):
    # Use the letter itself as the text for edge_tts to speak its name
    data = await get_audio_bytes(edge_tts.Communicate(letter.upper(), voice=VOICE, rate="-10%"))
    audio = load_audio_from_edge_tts_bytes(data)
    path = f"site/sounds/letters/{letter}.mp3"
    audio.export(path, format="mp3") # Export the cleaned audio
    return True

async def generate_all():
    all_words = [w for w, _ in confirmed_words]

    # Collect all unique letters for generating individual letter sounds and phonics
    all_unique_letters = set()
    for word, _ in confirmed_words:
        for char in word:
            all_unique_letters.add(char)
    sorted_unique_letters = sorted(list(all_unique_letters)) # Sort for consistent generation output

    print("Generating phonics sounds:")
    phonemes_saved_count = 0
    for char in sorted_unique_letters:
        if await save_phoneme_for_char(char):
            phonemes_saved_count += 1
            print(f"  {char} → phoneme sound saved")
    print(f"  Total {phonemes_saved_count} unique phoneme sounds generated.")

    print("\nGenerating full words:")
    for word in all_words:
        data = await get_audio_bytes(edge_tts.Communicate(word, voice=VOICE, rate="-10%"))
        audio = load_audio_from_edge_tts_bytes(data)
        with open(f"site/sounds/words/{word}.mp3", "wb") as f:
            audio.export(f, format="mp3")
        print(f"  {word}")

    print("\nGenerating individual letter sounds:")
    for letter in sorted_unique_letters:
        await save_letter_sound(letter)
        print(f"  {letter} → letter sound saved")


    print("\nAudio generation complete.")

await generate_all()

### Using Custom Phoneme Audio Files (Optional, for higher quality)

Generating accurate phoneme sounds with text-to-speech engines can be challenging, as they are typically optimized for natural-sounding speech rather than isolated phonetic segments. To ensure the highest quality phonics sounds, you can provide your own pre-recorded audio files.

Here's how to set this up:
1.  **Obtain Phoneme Audio Files:** Find or create high-quality MP3 audio files for each phoneme sound (e.g., the short 'a' sound, the 'b' sound, etc.).
2.  **Create Custom Folder:** Run the following code cell to create a dedicated folder for your custom phoneme sounds.
3.  **Upload & Place Files:** Upload your phoneme MP3 files into the newly created `site/sounds/phonics_custom/` folder. Name them using the single letter (e.g., `a.mp3`, `b.mp3`, `c.mp3`, etc.).

Once these files are in place, the system will automatically use them for the phonics sounds. If a custom file is not found for a specific letter, it will fall back to generating an approximate sound using `edge_tts`.

In [ ]:
# Create the directory for custom phoneme audio files
import os
os.makedirs("site/sounds/phonics_custom", exist_ok=True)
print("Directory 'site/sounds/phonics_custom' created (or already exists).")
print("You can now upload your custom phoneme MP3s here.")

Now I will update the audio generation logic to use your custom phoneme sounds if they exist, or fall back to `edge_tts` otherwise. This provides a robust solution for ensuring quality phonics audio.

In [ ]:
import edge_tts, asyncio, os, io, shutil
from pydub import AudioSegment
from pydub.silence import split_on_silence

VOICE = "en-US-JennyNeural"
PHONICS_CUSTOM_PATH = "site/sounds/phonics_custom/" # New path for custom phoneme audio

# IPA map (used as fallback if custom audio not provided)
IPA = {
    "a":"ah", "e":"eh", "i":"ih", "o":"aw", "u":"uh",
    "b":"buh",  "c":"kuh",  "d":"duh",  "f":"fuh",  "g":"guh",
    "h":"huh",  "j":"juh","k":"kuh",  "l":"luh",  "m":"muh",
    "n":"nuh",  "p":"puh",  "r":"ruh",  "s":"suh",  "t":"tuh",
    "v":"vuh",  "w":"wuh",  "x":"ks", "y":"yuh",  "z":"zuh",
}

os.makedirs("site/sounds/phonics", exist_ok=True)
os.makedirs("site/sounds/words",   exist_ok=True)
os.makedirs("site/sounds/letters", exist_ok=True)

async def get_audio_bytes(communicate):
    chunks = []
    async for chunk in communicate.stream():
        if chunk["type"] == "audio":
            chunks.append(chunk["data"])
    return b"".join(chunks)

def load_audio_from_edge_tts_bytes(data):
    temp_path = "temp_audio_data.mp3"
    with open(temp_path, "wb") as f:
        f.write(data)
    audio = AudioSegment.from_file(temp_path, format="mp3")
    os.remove(temp_path)
    return audio

async def save_phoneme_for_char(char):
    final_save_path = f"site/sounds/phonics/{char}.mp3"
    custom_file_path = os.path.join(PHONICS_CUSTOM_PATH, f"{char}.mp3")

    if os.path.exists(custom_file_path):
        shutil.copy(custom_file_path, final_save_path)
        print(f"  {char} → Using custom phoneme audio.")
        return True
    else:
        if char not in IPA:
            print(f"Warning: No IPA mapping or custom audio for character '{char}'. Skipping phoneme generation.")
            return False

        # Fallback to generating with edge_tts using English approximations
        data = await get_audio_bytes(edge_tts.Communicate(IPA[char], voice=VOICE))
        audio = load_audio_from_edge_tts_bytes(data)
        audio.export(final_save_path, format="mp3")
        print(f"  {char} → Generated fallback phoneme audio.")
        return True

async def save_letter_sound(letter):
    data = await get_audio_bytes(edge_tts.Communicate(letter.upper(), voice=VOICE, rate="-10%"))
    audio = load_audio_from_edge_tts_bytes(data)
    path = f"site/sounds/letters/{letter}.mp3"
    audio.export(path, format="mp3")
    return True

async def generate_all():
    all_words = [w for w, _ in confirmed_words]
    all_unique_letters = set()
    for word, _ in confirmed_words:
        for char in word:
            all_unique_letters.add(char)
    sorted_unique_letters = sorted(list(all_unique_letters))

    print("Generating phonics sounds:")
    phonemes_saved_count = 0
    for char in sorted_unique_letters:
        if await save_phoneme_for_char(char):
            phonemes_saved_count += 1
    print(f"  Total {phonemes_saved_count} unique phoneme sounds processed.")

    print("\nGenerating full words:")
    for word in all_words:
        data = await get_audio_bytes(edge_tts.Communicate(word, voice=VOICE, rate="-10%"))
        audio = load_audio_from_edge_tts_bytes(data)
        with open(f"site/sounds/words/{word}.mp3", "wb") as f:
            audio.export(f, format="mp3")
        print(f"  {word}")

    print("\nGenerating individual letter sounds:")
    for letter in sorted_unique_letters:
        await save_letter_sound(letter)
        print(f"  {letter} → letter sound saved")

    print("\nAudio generation complete.")

await generate_all()

## Step 9 — Build the HTML website

In [ ]:
import json
from collections import defaultdict

pages_dict = defaultdict(list)
for word, page_num in confirmed_words:
    pages_dict[page_num].append(word)

words_js = [{"word": w, "img": f"images/{w}.png", "page": p} for w, p in confirmed_words]
words_json = json.dumps(words_js, indent=2)

# Page section HTML
page_sections = ""
for pnum in sorted(pages_dict.keys()):
    vowels = " - ".join(w[1].upper() for w in pages_dict[pnum])
    page_sections += (
        f'\n  <div class="page-section">'
        f'\n    <div class="page-title">Short vowels: {vowels}</div>'
        f'\n    <div class="word-grid" id="page{pnum}"></div>'
        f'\n  </div>'
    )

CSS = """
    * { box-sizing: border-box; margin: 0; padding: 0; }
    body {
      font-family: 'Comic Sans MS', 'Chalkboard SE', cursive, sans-serif;
      background: linear-gradient(135deg, #FFF9F0 0%, #F0F8FF 100%);
      min-height: 100vh; padding: 20px;
    }
    h1 { text-align:center; font-size:2.2rem; color:#E05A2B; margin-bottom:8px; text-shadow:2px 2px 0 #FFD700; }
    .subtitle { text-align:center; color:#888; font-size:0.95rem; margin-bottom:28px; }
    .page-section { margin-bottom:40px; }
    .page-title { font-size:1.3rem; font-weight:bold; color:#5B4FCF; text-align:center; margin-bottom:14px; letter-spacing:1px; }
    .word-grid { display:flex; flex-direction:column; gap:12px; max-width:820px; margin:0 auto; }
    .word-row {
      display:grid; grid-template-columns:160px 1fr 180px; align-items:center; gap:12px;
      background:#fff; border-radius:18px; padding:12px 16px;
      box-shadow:0 4px 14px rgba(0,0,0,0.08); border:2px solid #e8e8e8; transition:border-color 0.2s;
    }
    .word-row:hover { border-color:#c5b8f5; }
    .pic-cell {
      display:flex; flex-direction:column; align-items:center; cursor:pointer;
      padding:8px; border-radius:14px; transition:background 0.18s,transform 0.15s;
      user-select:none; -webkit-user-select:none;
    }
    .pic-cell:hover  { background:#FFF3CD; transform:scale(1.04); }
    .pic-cell:active { transform:scale(0.97); }
    .pic-cell.reading { background:#D4EDDA; animation:glow 0.7s infinite alternate; }
    @keyframes glow { from{box-shadow:0 0 0 0 rgba(40,167,69,0.4);} to{box-shadow:0 0 0 10px rgba(40,167,69,0);} }
    .pic-cell img { width:120px; height:90px; object-fit:contain; }
    .pic-hint { font-size:0.7rem; color:#aaa; margin-top:4px; }
    .letter-boxes { display:flex; justify-content:center; align-items:center; gap:8px; }
    .letter-box {
      width:64px; height:64px; border-radius:12px; border:3px solid #bbb;
      display:flex; align-items:center; justify-content:center;
      font-size:2.2rem; font-weight:bold; color:#333; transition:all 0.35s cubic-bezier(.34,1.56,.64,1);
    }
    .vowel-box { background:#FFD700; border-color:#E6A800; color:#7A5200; }
    .consonant-box { background:#E8F4FD; border-color:#90CAF9; }
    .consonant-box.hidden-letter { background:#F5F5F5; border-color:#ddd; color:transparent; }
    .consonant-box.revealed { background:#E8F4FD; border-color:#42A5F5; animation:popIn 0.4s cubic-bezier(.34,1.56,.64,1); }
    @keyframes popIn { from{transform:scale(0.5);opacity:0.2;} to{transform:scale(1);opacity:1;} }
    .btn-cell { display:flex; flex-direction:column; gap:8px; align-items:center; }
    .btn { width:130px; padding:10px 0; border:none; border-radius:30px; font-family:inherit; font-size:1rem; font-weight:bold; cursor:pointer; transition:transform 0.12s,box-shadow 0.12s; }
    .btn:active { transform:scale(0.95); }
    .btn-show { background:linear-gradient(135deg,#43D08A,#1DAF66); color:white; box-shadow:0 4px 12px rgba(29,175,102,0.35); }
    .btn-show:hover { box-shadow:0 6px 18px rgba(29,175,102,0.5); transform:translateY(-2px); }
    .btn-show:disabled { background:#aaa; box-shadow:none; cursor:not-allowed; transform:none; }
    .btn-reset { background:linear-gradient(135deg,#FF7A7A,#E74C3C); color:white; box-shadow:0 4px 12px rgba(231,76,60,0.35); }
    .btn-reset:hover { box-shadow:0 6px 18px rgba(231,76,60,0.5); transform:translateY(-2px); }
    .sound-wave { display:inline-flex; align-items:flex-end; gap:2px; height:18px; margin-left:6px; vertical-align:middle; opacity:0; transition:opacity 0.2s; }
    .sound-wave.active { opacity:1; }
    .sound-wave span { display:block; width:3px; background:#43D08A; border-radius:2px; animation:wave 0.6s ease-in-out infinite alternate; }
    .sound-wave span:nth-child(1){height:6px; animation-delay:0s;}
    .sound-wave span:nth-child(2){height:14px;animation-delay:0.1s;}
    .sound-wave span:nth-child(3){height:10px;animation-delay:0.2s;}
    .sound-wave span:nth-child(4){height:18px;animation-delay:0.15s;}
    .sound-wave span:nth-child(5){height:8px; animation-delay:0.05s;}
    @keyframes wave { from{transform:scaleY(0.4);} to{transform:scaleY(1);} }
    footer { text-align:center; color:#bbb; font-size:0.8rem; margin-top:30px; padding-bottom:20px; }
"""

JS = f"""
const WORDS = {words_json};
const PHONICS_PATH = 'sounds/phonics/'; // Renamed PH to PHONICS_PATH for clarity
const WORDS_PATH = 'sounds/words/';     // Renamed WD to WORDS_PATH for clarity
const LETTERS_PATH = 'sounds/letters/'; // New path for individual letter sounds
const EXT = '.mp3';
const cache = {{}};
function getAudio(p){{ if(!cache[p]) cache[p]=new Audio(p); return cache[p]; }}
function playAudio(p){{
  return new Promise(res=>{{
    const a=getAudio(p); a.currentTime=0;
    const done=()=>{{
    a.removeEventListener('ended',done);res();}};
    a.addEventListener('ended',done); a.play().catch(()=>res());
  }});
}}
function pause(ms){{return new Promise(r=>setTimeout(r,ms));}};
function showWave(el,on){{el.classList.toggle('active',on);}}
function buildRow(d,idx){{
  const {{word,img}}=d; const id=`row-${{idx}}`;
  const row=document.createElement('div'); row.className='word-row';
  row.innerHTML=`
    <div class="pic-cell" id="${{id}}-pic" title="Click to hear!">
      <img src="${{img}}" alt="${{word}}">
      <span class="pic-hint">Click to listen
        <span class="sound-wave" id="${{id}}-wave"><span></span><span></span><span></span><span></span><span></span></span>
      </span>
    </div>
    <div class="letter-boxes">
      <div class="letter-box consonant-box hidden-letter" id="${{id}}-first">${{word[0]}}</div>
      <div class="letter-box vowel-box">${{word[1]}}</div>
      <div class="letter-box consonant-box hidden-letter" id="${{id}}-last">${{word[2]}}</div>
    </div>
    <div class="btn-cell">
      <button class="btn btn-show"  id="${{id}}-show-btn">Show</button>
      <button class="btn btn-reset" id="${{id}}-reset-btn">Reset</button>
    </div>`;
  return row;
}}
function wireRow(d,idx){{
  const {{word}}=d; const id=`row-${{idx}}`;
  const pic=document.getElementById(`${{id}}-pic`);
  const fb=document.getElementById(`${{id}}-first`);
  const lb=document.getElementById(`${{id}}-last`);
  const sb=document.getElementById(`${{id}}-show-btn`);
  const rb=document.getElementById(`${{id}}-reset-btn`);
  const wv=document.getElementById(`${{id}}-wave`);

  const isVowel = (char) => ['a', 'e', 'i', 'o', 'u'].includes(char.toLowerCase());

  // Image click handler (play phonics and whole word)
  pic.addEventListener('click',async()=>{{
    if(pic.classList.contains('reading'))return;
    pic.classList.add('reading');showWave(wv,true);
    await playAudio(PHONICS_PATH+word[0]+EXT);await pause(100);
    await playAudio(PHONICS_PATH+word[1]+EXT);await pause(100);
    await playAudio(PHONICS_PATH+word[2]+EXT);await pause(175);
    await playAudio(WORDS_PATH+word+EXT);
    showWave(wv,false);pic.classList.remove('reading');
  }});

  // "Show" button handler
  sb.addEventListener('click',async()=>{{
    sb.disabled=true; // Disable button during animation

    // First letter
    await playAudio(LETTERS_PATH+word[0]+EXT); // Play letter name sound
    if (!isVowel(word[0])) {{
      fb.textContent=word[0];
      fb.classList.remove('hidden-letter');
      fb.classList.add('revealed');
    }}
    await pause(2); // Changed from 55

    // Second letter (vowel) - always visible, just play sound
    await playAudio(LETTERS_PATH+word[1]+EXT);
    await pause(2); // Changed from 55

    // Third letter
    await playAudio(LETTERS_PATH+word[2]+EXT); // Play letter name sound
    if (!isVowel(word[2])) {{
      lb.textContent=word[2];
      lb.classList.remove('hidden-letter');
      lb.classList.add('revealed');
    }}
    await pause(2); // Changed from 88

    // Play whole word
    await playAudio(WORDS_PATH+word+EXT);
    sb.disabled=false; // Re-enable button
  }});

  // "Reset" button handler
  rb.addEventListener('click',()=>{{
    Object.values(cache).forEach(a=>{{
    a.pause();a.currentTime=0;}});
    // Reset first consonant box
    if (!isVowel(word[0])) {{
      fb.classList.add('hidden-letter');
      fb.classList.remove('revealed');
    }}
    // Reset last consonant box
    if (!isVowel(word[2])) {{
      lb.classList.add('hidden-letter');
      lb.classList.remove('revealed');
    }}
    sb.disabled=false;
    pic.classList.remove('reading');
    showWave(wv,false);
  }});
}}
const containers=Object.fromEntries([...new Set(WORDS.map(w=>w.page))].map(p=>[p,`page${{p}}`]));
WORDS.forEach((d,i)=>{{
const c=document.getElementById(containers[d.page]);if(c){{
    c.appendChild(buildRow(d,i));wireRow(d,i);}}}});
"""

html = f"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>CVC Word Practice</title>
  <style>{CSS}</style>
</head>
<body>
  <h1>CVC Word Practice</h1>
  <p class="subtitle">Click a picture to hear the sounds - Use Show and Reset to practise writing</p>
  {page_sections}
  <footer>CVC First and Last Letter Practice</footer>
<script>{JS}</script>
</body>
</html>"""

with open("site/index.html", "w") as f:
    f.write(html)
print("site/index.html written successfully.")

## Step 9b — Build HTML with SAS Token (for shared links)

This step builds the HTML website exactly the same way as Step 9, but modifies the JavaScript to dynamically append a `sas_token` (Shared Access Signature token) to all asset URLs (images, phonics, words, letters). This is useful when sharing the generated website from a cloud storage service (like Azure Blob Storage) that requires a SAS token for asset access. The `sas_token` will be read from the browser's URL query parameters.

In [ ]:
import json
from collections import defaultdict

pages_dict = defaultdict(list)
for word, page_num in confirmed_words:
    pages_dict[page_num].append(word)

words_js = [{"word": w, "img": f"images/{w}.png", "page": p} for w, p in confirmed_words]
words_json = json.dumps(words_js, indent=2)

# Page section HTML
page_sections = ""
for pnum in sorted(pages_dict.keys()):
    vowels = " - ".join(w[1].upper() for w in pages_dict[pnum])
    page_sections += f"""
  <div class="page-section">
    <div class="page-title">Short vowels: {vowels}</div>
    <div class="word-grid" id="page{pnum}"></div>
  </div>
"""

CSS = """
    * { box-sizing: border-box; margin: 0; padding: 0; }
    body {
      font-family: 'Comic Sans MS', 'Chalkboard SE', cursive, sans-serif;
      background: linear-gradient(135deg, #FFF9F0 0%, #F0F8FF 100%);
      min-height: 100vh; padding: 20px;
    }
    h1 { text-align:center; font-size:2.2rem; color:#E05A2B; margin-bottom:8px; text-shadow:2px 2px 0 #FFD700; }
    .subtitle { text-align:center; color:#888; font-size:0.95rem; margin-bottom:28px; }
    .page-section { margin-bottom:40px; }
    .page-title { font-size:1.3rem; font-weight:bold; color:#5B4FCF; text-align:center; margin-bottom:14px; letter-spacing:1px; }
    .word-grid { display:flex; flex-direction:column; gap:12px; max-width:820px; margin:0 auto; }
    .word-row {
      display:grid; grid-template-columns:160px 1fr 180px; align-items:center; gap:12px;
      background:#fff; border-radius:18px; padding:12px 16px;
      box-shadow:0 4px 14px rgba(0,0,0,0.08); border:2px solid #e8e8e8; transition:border-color 0.2s;
    }
    .word-row:hover { border-color:#c5b8f5; }
    .pic-cell {
      display:flex; flex-direction:column; align-items:center; cursor:pointer;
      padding:8px; border-radius:14px; transition:background 0.18s,transform 0.15s;
      user-select:none; -webkit-user-select:none;
    }
    .pic-cell:hover  { background:#FFF3CD; transform:scale(1.04); }
    .pic-cell:active { transform:scale(0.97); }
    .pic-cell.reading { background:#D4EDDA; animation:glow 0.7s infinite alternate; }
    @keyframes glow { from{box-shadow:0 0 0 0 rgba(40,167,69,0.4);} to{box-shadow:0 0 0 10px rgba(40,167,69,0);} }
    .pic-cell img { width:120px; height:90px; object-fit:contain; }
    .pic-hint { font-size:0.7rem; color:#aaa; margin-top:4px; }
    .letter-boxes { display:flex; justify-content:center; align-items:center; gap:8px; }
    .letter-box {
      width:64px; height:64px; border-radius:12px; border:3px solid #bbb;
      display:flex; align-items:center; justify-content:center;
      font-size:2.2rem; font-weight:bold; color:#333; transition:all 0.35s cubic-bezier(.34,1.56,.64,1);
    }
    .vowel-box { background:#FFD700; border-color:#E6A800; color:#7A5200; }
    .consonant-box { background:#E8F4FD; border-color:#90CAF9; }
    .consonant-box.hidden-letter { background:#F5F5F5; border-color:#ddd; color:transparent; }
    .consonant-box.revealed { background:#E8F4FD; border-color:#42A5F5; animation:popIn 0.4s cubic-bezier(.34,1.56,.64,1); }
    @keyframes popIn { from{transform:scale(0.5);opacity:0.2;} to{transform:scale(1);opacity:1;} }
    .btn-cell { display:flex; flex-direction:column; gap:8px; align-items:center; }
    .btn { width:130px; padding:10px 0; border:none; border-radius:30px; font-family:inherit; font-size:1rem; font-weight:bold; cursor:pointer; transition:transform 0.12s,box-shadow 0.12s; }
    .btn:active { transform:scale(0.95); }
    .btn-show { background:linear-gradient(135deg,#43D08A,#1DAF66); color:white; box-shadow:0 4px 12px rgba(29,175,102,0.35); }
    .btn-show:hover { box-shadow:0 6px 18px rgba(29,175,102,0.5); transform:translateY(-2px); }
    .btn-show:disabled { background:#aaa; box-shadow:none; cursor:not-allowed; transform:none; }
    .btn-reset { background:linear-gradient(135deg,#FF7A7A,#E74C3C); color:white; box-shadow:0 4px 12px rgba(231,76,60,0.35); }
    .btn-reset:hover { box-shadow:0 6px 18px rgba(231,76,60,0.5); transform:translateY(-2px); }
    .sound-wave { display:inline-flex; align-items:flex-end; gap:2px; height:18px; margin-left:6px; vertical-align:middle; opacity:0; transition:opacity 0.2s; }
    .sound-wave.active { opacity:1; }
    .sound-wave span { display:block; width:3px; background:#43D08A; border-radius:2px; animation:wave 0.6s ease-in-out infinite alternate; }
    .sound-wave span:nth-child(1){height:6px; animation-delay:0s;}
    .sound-wave span:nth-child(2){height:14px;animation-delay:0.1s;}
    .sound-wave span:nth-child(3){height:10px;animation-delay:0.2s;}
    .sound-wave span:nth-child(4){height:18px;animation-delay:0.15s;}
    .sound-wave span:nth-child(5){height:8px; animation-delay:0.05s;}
    @keyframes wave { from{transform:scaleY(0.4);} to{transform:scaleY(1);} }
    footer { text-align:center; color:#bbb; font-size:0.8rem; margin-top:30px; padding-bottom:20px; }
"""

JS = f"""
const WORDS = {words_json};
const PHONICS_PATH_BASE = 'sounds/phonics/';
const WORDS_PATH_BASE = 'sounds/words/';
const LETTERS_PATH_BASE = 'sounds/letters/';
const EXT = '.mp3';
const cache = {{}};

// Extract SAS token from URL if present (captures the full query string)
const sasToken = window.location.search;
console.log('Detected full query string (SAS Token if present):', sasToken); // Debugging line

function getAudio(p){{
  const fullPath = p + sasToken; // Append sasToken here
  console.log('Constructed audio URL:', fullPath); // Debugging line
  if(!cache[fullPath]) cache[fullPath]=new Audio(fullPath);
  return cache[fullPath];
}}
function playAudio(p){{
  return new Promise(res=>{{
    const a=getAudio(p); a.currentTime=0;
    const done=()=>{{
    a.removeEventListener('ended',done);res();}};
    a.addEventListener('ended',done); a.play().catch(()=>res());
  }});
}}
function pause(ms){{return new Promise(r=>setTimeout(r,ms));}};
function showWave(el,on){{el.classList.toggle('active',on);}};
function buildRow(d,idx){{
  const {{word,img}}=d; const id=`row-${{idx}}`;
  const row=document.createElement('div'); row.className='word-row';
  // Append sasToken to img src here
  const imgSrc = img + sasToken;
  console.log('Constructed image URL:', imgSrc); // Debugging line
  row.innerHTML=`
    <div class="pic-cell" id="${{id}}-pic" title="Click to hear!">
      <img src="${{imgSrc}}" alt="${{word}}">
      <span class="pic-hint">Click to listen
        <span class="sound-wave" id="${{id}}-wave"><span></span><span></span><span></span><span></span><span></span></span>
      </span>
    </div>
    <div class="letter-boxes">
      <div class="letter-box consonant-box hidden-letter" id="${{id}}-first">${{word[0]}}</div>
      <div class="letter-box vowel-box">${{word[1]}}</div>
      <div class="letter-box consonant-box hidden-letter" id="${{id}}-last">${{word[2]}}</div>
    </div>
    <div class="btn-cell">
      <button class="btn btn-show"  id="${{id}}-show-btn">Show</button>
      <button class="btn btn-reset" id="${{id}}-reset-btn">Reset</button>
    </div>`;
  return row;
}}
function wireRow(d,idx){{
  const {{word}}=d; const id=`row-${{idx}}`;
  const pic=document.getElementById(`${{id}}-pic`);
  const fb=document.getElementById(`${{id}}-first`);
  const lb=document.getElementById(`${{id}}-last`);
  const sb=document.getElementById(`${{id}}-show-btn`);
  const rb=document.getElementById(`${{id}}-reset-btn`);
  const wv=document.getElementById(`${{id}}-wave`);

  const isVowel = (char) => ['a', 'e', 'i', 'o', 'u'].includes(char.toLowerCase());

  // Image click handler (play phonics and whole word)
  pic.addEventListener('click',async()=>{{
    if(pic.classList.contains('reading'))return;
    pic.classList.add('reading');showWave(wv,true);
    await playAudio(PHONICS_PATH_BASE+word[0]+EXT);await pause(100);
    await playAudio(PHONICS_PATH_BASE+word[1]+EXT);await pause(100);
    await playAudio(PHONICS_PATH_BASE+word[2]+EXT);await pause(175);
    await playAudio(WORDS_PATH_BASE+word+EXT);
    showWave(wv,false);pic.classList.remove('reading');
  }});

  // "Show" button handler
  sb.addEventListener('click',async()=>{{
    sb.disabled=true; // Disable button during animation

    // First letter
    await playAudio(LETTERS_PATH_BASE+word[0]+EXT); // Play letter name sound
    if (!isVowel(word[0])) {{
      fb.textContent=word[0];
      fb.classList.remove('hidden-letter');
      fb.classList.add('revealed');
    }}
    await pause(2);

    // Second letter (vowel) - always visible, just play sound
    await playAudio(LETTERS_PATH_BASE+word[1]+EXT);
    await pause(2);

    // Third letter
    await playAudio(LETTERS_PATH_BASE+word[2]+EXT); // Play letter name sound
    if (!isVowel(word[2])) {{
      lb.textContent=word[2];
      lb.classList.remove('hidden-letter');
      lb.classList.add('revealed');
    }}
    await pause(2);

    // Play whole word
    await playAudio(WORDS_PATH_BASE+word+EXT);
    sb.disabled=false; // Re-enable button
  }});

  // "Reset" button handler
  rb.addEventListener('click',()=>{{
    Object.values(cache).forEach(a=>{{
    a.pause();a.currentTime=0;}});
    // Reset first consonant box
    if (!isVowel(word[0])) {{
      fb.classList.add('hidden-letter');
      fb.classList.remove('revealed');
    }}
    // Reset last consonant box
    if (!isVowel(word[2])) {{
      lb.classList.add('hidden-letter');
      lb.classList.remove('revealed');
    }}
    sb.disabled=false;
    pic.classList.remove('reading');
    showWave(wv,false);
  }});
}}
const containers=Object.fromEntries([...new Set(WORDS.map(w=>w.page))].map(p=>[p,`page${{p}}`]));
WORDS.forEach((d,i)=>{{
const c=document.getElementById(containers[d.page]);if(c){{
    c.appendChild(buildRow(d,i));wireRow(d,i);}}}});
"""

html = f"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>CVC Word Practice</title>
  <style>{CSS}</style>
</head>
<body>
  <h1>CVC Word Practice</h1>
  <p class="subtitle">Click a picture to hear the sounds - Use Show and Reset to practise writing</p>
  {page_sections}
  <footer>CVC First and Last Letter Practice</footer>
<script>{JS}</script>
</body>
</html>"""

with open("site/index_sas.html", "w") as f:
    f.write(html)
print("site/index_sas.html written successfully. This version includes SAS token handling.")

## Step 10 — Package and download

In [ ]:
import zipfile, os
from google.colab import files as colab_files

base = os.path.splitext(os.path.basename(PDF_PATH))[0]
zip_name = f"{base}_website.zip"

print(f"Creating {zip_name} ...")
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, file_list in os.walk("site"):
        for fname in file_list:
            full = os.path.join(root, fname)
            zf.write(full, os.path.relpath(full, "."))

size_kb = os.path.getsize(zip_name) // 1024
print(f"Ready: {zip_name} ({size_kb} KB)")
colab_files.download(zip_name)
print("\nDone! Unzip the file and open site/index.html in any browser.")
